In [1]:
#imports

import pandas as pd 
import numpy as np 
import datetime

In [2]:
# Loading Datasets

stock = pd.read_csv("HistoricalData_GOOGLE.csv")
treasury = pd.read_csv("HistoricalData_10YrTreasury.csv")

In [3]:
#Standardizing Alphabet(Google) Stock Price Dataset

stock["Date"] = pd.to_datetime(stock["Date"])
stock["Close/Last"] = stock["Close/Last"].replace(r"[\$,]", "", regex=True).astype(float)
stock = stock.drop(columns = ["Volume", "Open", "High", "Low"])
stock = stock.rename(columns = {"Close/Last": "Price"})
print(stock)

           Date     Price
0    2026-03-11  308.4200
1    2026-03-10  306.9300
2    2026-03-09  306.0100
3    2026-03-06  298.3000
4    2026-03-05  300.9100
...         ...       ...
1250 2021-03-18  101.8110
1251 2021-03-17  104.5540
1252 2021-03-16  104.6260
1253 2021-03-15  103.3245
1254 2021-03-12  103.0960

[1255 rows x 2 columns]


In [4]:
# Identifying variable formats

stock.dtypes

Date     datetime64[ns]
Price           float64
dtype: object

In [5]:
#Checking for missing values

stock.isnull().sum()

Date     0
Price    0
dtype: int64

In [6]:
# Identifying variable formats

treasury.dtypes

observation_date     object
DGS10               float64
dtype: object

In [7]:
#Checking for missing values

treasury.isnull().sum()

observation_date     0
DGS10               55
dtype: int64

In [8]:
#Standardizing US 10-year Treasury Yield Dataset

treasury = treasury.rename(columns = {"observation_date": "Date", "DGS10": "Rate"})
treasury["Date"] = pd.to_datetime(treasury["Date"])

In [9]:
# Merging both datasets

merged_dataset = pd.merge(stock, treasury, on="Date", how="inner")

In [10]:
#Filling missing values

merged_dataset["Rate"] = merged_dataset["Rate"].ffill()
print(merged_dataset)

           Date     Price  Rate
0    2026-03-11  308.4200  4.21
1    2026-03-10  306.9300  4.15
2    2026-03-09  306.0100  4.12
3    2026-03-06  298.3000  4.15
4    2026-03-05  300.9100  4.13
...         ...       ...   ...
1250 2021-03-18  101.8110  1.71
1251 2021-03-17  104.5540  1.63
1252 2021-03-16  104.6260  1.62
1253 2021-03-15  103.3245  1.62
1254 2021-03-12  103.0960  1.64

[1255 rows x 3 columns]


In [11]:
#Checking for missing values

merged_dataset.isnull().sum()

Date     0
Price    0
Rate     0
dtype: int64

In [12]:
#Calculating Daily Price Change (Value in Percent)

merged_dataset["Return"] = merged_dataset["Price"].pct_change() *100
merged_dataset["Return"]

0            NaN
1      -0.483107
2      -0.299743
3      -2.519526
4       0.874958
          ...   
1250   -0.341621
1251    2.694208
1252    0.068864
1253   -1.243955
1254   -0.221148
Name: Return, Length: 1255, dtype: float64

In [13]:
#Calculating Daily Average Return

avg_return = merged_dataset["Return"].mean()

print(avg_return)

-0.06887101248234782


In [14]:
#Calculating Daily Volatility

volatility = merged_dataset["Return"].std()

print(volatility)

1.9244043616567654


In [15]:
#Calculating Annual Volatility

annual_vol = merged_dataset["Return"].std() * (252 ** 0.5)

print(annual_vol)

30.548972177230823


In [16]:
#Combining all the metrics in one table (Value in Percent)

metrics = pd.DataFrame({
    "Metric": ["Avg Daily Return", "Daily Volatility", "Annual Volatility"],
    "Value": [avg_return, volatility, annual_vol]})

print(metrics)

              Metric      Value
0   Avg Daily Return  -0.068871
1   Daily Volatility   1.924404
2  Annual Volatility  30.548972
